# OCR de patentes

Este notebook se construyó **desde cero** para demostrar cómo generar recortes de patentes a partir de las anotaciones de ground truth, aplicar distintos preprocesados y evaluar distintos motores de OCR. Está pensado para un entorno Windows con estructura de carpetas:

- `video01/video.mp4`: vídeo de entrada.
- `video01/vehicles.xml`: anotaciones de vehículos con etiquetas `plate="True"` para las patentes visibles.
- `lanes.xml`: definición de carriles (no se usa aquí).

El flujo general es:

1. Leer el XML y construir un `DataFrame` con las regiones de patente (solo cuando `plate="True"`).
2. Visualizar los recortes de patente para inspeccionar su calidad.
3. Definir varias funciones de preprocesado (incluyendo mejoras como CLAHE y filtrado bilateral) para preparar las imágenes antes de OCR.
4. Ejecutar Tesseract (si está disponible) y EasyOCR sobre los recortes y recopilar los resultados en un DataFrame.
5. Calcular algunas métricas sencillas (porcentaje de textos no vacíos y que cumplen un patrón de matrícula).

Nota: este notebook asume que tienes instalados `opencv-python`, `pandas`, `numpy`, `pytesseract`, `easyocr` y `matplotlib`. Si Tesseract no está instalado o accesible, se omitirá esa parte automáticamente.


In [ ]:

import cv2
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import importlib

# Intenta importar pytesseract y easyocr
try:
    import pytesseract
except ImportError:
    pytesseract = None

try:
    import easyocr
except ImportError:
    easyocr = None

# Configurar uso de Tesseract si está disponible
if pytesseract is not None:
    # Establecer la ruta al ejecutable de Tesseract si es necesario
    # Descomenta y ajusta la ruta en Windows, p. ej.:
    # pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
    pass

print("pytesseract disponible:", pytesseract is not None)
print("easyocr disponible:", easyocr is not None)

# Directorios base
BASE_DIR = Path(".").resolve()
VIDEO_PATH = BASE_DIR / "video01" / "video.mp4"
XML_PATH   = BASE_DIR / "video01" / "vehicles.xml"

print("BASE_DIR:", BASE_DIR)
print("VIDEO_PATH:", VIDEO_PATH)
print("XML_PATH:", XML_PATH)


In [ ]:

def build_gt_plates_df(xml_path: Path) -> pd.DataFrame:
    """
    Lee el XML de ground truth y arma un DataFrame con las bounding boxes
    de las patentes (cuando plate="True"). También incluye información
    de velocidad de radar si está disponible.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    entries = []
    for idx, veh in enumerate(root.findall(".//vehicle")):
        has_plate = veh.get("plate", "False").lower() == "true"
        if not has_plate:
            continue
        frame = int(veh.get("iframe"))
        lane = int(veh.get("lane"))
        # región de la matrícula
        region = veh.find("region")
        x = int(region.get("x"))
        y = int(region.get("y"))
        w = int(region.get("w"))
        h = int(region.get("h"))
        x1, y1 = x, y
        x2, y2 = x + w, y + h

        # datos de radar (pueden no existir)
        radar_tag = veh.find("radar")
        if radar_tag is not None:
            radar_start = int(radar_tag.get("frame_start"))
            radar_end   = int(radar_tag.get("frame_end"))
            radar_speed = float(radar_tag.get("speed"))
        else:
            radar_start = np.nan
            radar_end   = np.nan
            radar_speed = np.nan

        entries.append({
            'frame': frame,
            'xml_idx': idx,
            'lane': lane,
            'x1': x1,
            'y1': y1,
            'x2': x2,
            'y2': y2,
            'w': w,
            'h': h,
            'radar_frame_start': radar_start,
            'radar_frame_end': radar_end,
            'radar_speed': radar_speed
        })

    df = pd.DataFrame(entries)
    return df

# Construir el DataFrame de GT
df_gt = build_gt_plates_df(XML_PATH)
print(f"Número de matrículas con GT: {len(df_gt)}")
df_gt.head()


In [ ]:

def show_gt_plate(frame_idx: int, df_gt: pd.DataFrame, video_path: Path, figsize=(12, 7)):
    """
    Muestra el frame original y dibuja las bboxes de patente GT para ese frame.
    """
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_idx >= total_frames:
        print(f"Frame {frame_idx} fuera de rango. Total de frames: {total_frames}")
        cap.release()
        return
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        print(f"No se pudo leer el frame {frame_idx}")
        return

    # Filtrar DataFrame por este frame
    subset = df_gt[df_gt['frame'] == frame_idx]

    # Mostrar imagen
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.axis('off')

    for _, row in subset.iterrows():
        rect = patches.Rectangle((row['x1'], row['y1']), row['w'], row['h'], linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        ax.text(row['x1'], row['y1'] - 5, f"Lane {row['lane']}", color='lime', fontsize=10, weight='bold')

    plt.show()

# Ejemplo: mostrar la primera matrícula
if len(df_gt) > 0:
    show_gt_plate(int(df_gt.iloc[0]['frame']), df_gt, VIDEO_PATH)


In [ ]:

def build_plate_crops(df_gt: pd.DataFrame, video_path: Path, margin: float = 0.2, max_samples: int | None = None) -> pd.DataFrame:
    """
    A partir del df de GT arma un DataFrame con los recortes de las patentes.
    margin define cuánto margen extra incluir alrededor de la caja (proporción del tamaño original).
    max_samples sirve para limitar el número de recortes (útil para pruebas).
    """
    cap = cv2.VideoCapture(str(video_path))
    results = []
    for idx, (_, row) in enumerate(df_gt.iterrows()):
        if max_samples is not None and idx >= max_samples:
            break
        frame_idx = int(row['frame'])
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            continue
        x1, y1, x2, y2 = int(row['x1']), int(row['y1']), int(row['x2']), int(row['y2'])
        # aplicar margen
        w, h = row['w'], row['h']
        dx = int(w * margin)
        dy = int(h * margin)
        x1_m = max(0, x1 - dx)
        y1_m = max(0, y1 - dy)
        x2_m = min(frame.shape[1], x2 + dx)
        y2_m = min(frame.shape[0], y2 + dy)

        crop = frame[y1_m:y2_m, x1_m:x2_m]
        results.append({
            'idx': idx,
            'frame': frame_idx,
            'lane': row['lane'],
            'x1': x1_m,
            'y1': y1_m,
            'x2': x2_m,
            'y2': y2_m,
            'crop': crop
        })
    cap.release()
    df_crops = pd.DataFrame(results)
    print(f"Cantidad de recortes: {len(df_crops)}")
    return df_crops

# Crear recortes de ejemplo (limitando a 50 por rapidez)
df_crops = build_plate_crops(df_gt, VIDEO_PATH, margin=0.2, max_samples=50)
df_crops.head()


In [ ]:

def show_plate_crops_grid(df_crops: pd.DataFrame, n: int = 8, figsize=(12, 6)):
    """
    Muestra n recortes de patente en un grid para inspeccionar su apariencia.
    """
    n = min(n, len(df_crops))
    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    for i, ax in enumerate(axes.flat):
        ax.axis('off')
        if i < n:
            crop = df_crops.iloc[i]['crop']
            ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            ax.set_title(f"idx={df_crops.iloc[i]['idx']} frame={df_crops.iloc[i]['frame']}")
    plt.tight_layout()
    plt.show()

# Mostrar 8 recortes para comprobar
show_plate_crops_grid(df_crops, n=8)


In [ ]:

def preproc_none(crop_bgr: np.ndarray) -> np.ndarray:
    """Devuelve el recorte en color BGR sin cambios."""
    return crop_bgr

def preproc_adaptive(crop_bgr: np.ndarray, scale: int = 3) -> np.ndarray:
    """
    Escala la imagen, pasa a gris, aplica desenfoque Gaussiano y umbral adaptativo.
    Devuelve una imagen binaria.
    """
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    gray = cv2.resize(gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    th = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11, 2
    )
    return th

def preproc_morpho(crop_bgr: np.ndarray, scale: int = 3) -> np.ndarray:
    """
    Escala la imagen, pasa a gris, aplica umbral de Otsu y cierre morfológico.
    """
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    gray = cv2.resize(gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((3, 3), np.uint8)
    closing = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=1)
    return closing

def preproc_clahe(crop_bgr: np.ndarray, scale: int = 3) -> np.ndarray:
    """
    Escala la imagen, pasa a gris, aplica CLAHE, filtrado bilateral y umbral de Otsu.
    """
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    gray = cv2.resize(gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)
    # CLAHE para mejorar contraste
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    eq = clahe.apply(gray)
    # Filtrado bilateral para reducir ruido conservando bordes
    denoise = cv2.bilateralFilter(eq, d=9, sigmaColor=75, sigmaSpace=75)
    # Otsu
    _, th = cv2.threshold(denoise, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return th

def preproc_bilateral_adaptive(crop_bgr: np.ndarray, scale: int = 3) -> np.ndarray:
    """
    Escala la imagen, pasa a gris, aplica filtrado bilateral y umbral adaptativo.
    """
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    gray = cv2.resize(gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)
    denoise = cv2.bilateralFilter(gray, d=9, sigmaColor=75, sigmaSpace=75)
    th = cv2.adaptiveThreshold(
        denoise, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        15, 2
    )
    return th

PREPROCS = {
    'none': preproc_none,
    'adaptive': preproc_adaptive,
    'morpho': preproc_morpho,
    'clahe': preproc_clahe,
    'bilateral_adaptive': preproc_bilateral_adaptive,
}


In [ ]:

import re

# Regex muy laxo: 6 a 8 caracteres alfanuméricos (caso simple de patente)
plate_regex = re.compile(r'^[A-Z0-9]{6,8}$')

def clean_plate_text(text: str) -> str:
    """Deja solo caracteres alfanuméricos en mayúsculas."""
    return ''.join(ch for ch in text.upper() if ch.isalnum())

def looks_like_plate(text: str) -> bool:
    """Heurística simple para ver si el string parece una matrícula."""
    return bool(plate_regex.match(text))

def run_tesseract(img: np.ndarray) -> str:
    """Ejecuta Tesseract sobre una imagen en escala de grises o binaria.
    Ajusta los parámetros para matrículas. Devuelve el texto limpio.
    """
    if pytesseract is None:
        return ''
    # Convertir a PIL o pasar directamente
    config = '--oem 1 --psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
    try:
        text = pytesseract.image_to_string(img, config=config)
    except pytesseract.pytesseract.TesseractNotFoundError:
        return ''
    return clean_plate_text(text)

def run_easyocr(img: np.ndarray, reader=None) -> str:
    """Ejecuta EasyOCR sobre una imagen y devuelve el texto limpio."""
    if easyocr is None:
        return ''
    if reader is None:
        reader = easyocr.Reader(['en'], gpu=False)
    # EasyOCR acepta imágenes en formato RGB
    if len(img.shape) == 2:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    else:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = reader.readtext(img_rgb, detail=0, paragraph=False)
    if not result:
        return ''
    # concatenar y limpiar
    return clean_plate_text(''.join(result))

# Función para ejecutar OCRs sobre todos los recortes y recolectar resultados
def evaluate_ocr(df_crops: pd.DataFrame, preprocs: dict, methods: list | None = None, max_samples: int | None = None) -> pd.DataFrame:
    """
    Aplica los preprocesados y métodos de OCR especificados sobre cada recorte.
    Devuelve un DataFrame con una fila por recorte y método.
    """
    records = []
    # Crear lector de easyocr una sola vez si se necesita
    reader = None
    if easyocr is not None and (methods is None or any('easy' in m for m in methods)):
        reader = easyocr.Reader(['en'], gpu=False)
    for i, row in enumerate(df_crops.itertuples(index=False)):
        if max_samples is not None and i >= max_samples:
            break
        crop = row.crop
        for pname, pfunc in preprocs.items():
            pre_img = pfunc(crop)
            # Asegurarse de que es imagen binaria/gris para tesseract
            # Ejecutar Tesseract
            if methods is None or 'tesseract' in methods:
                text_tess = run_tesseract(pre_img)
                records.append({
                    'idx': row.idx,
                    'frame': row.frame,
                    'method': f'tesseract_{pname}',
                    'text': text_tess,
                    'len': len(text_tess),
                    'non_empty': len(text_tess) > 0,
                    'looks_plate': looks_like_plate(text_tess)
                })
            # Ejecutar EasyOCR
            if methods is None or 'easyocr' in methods:
                text_easy = run_easyocr(pre_img if len(pre_img.shape)==2 else pre_img, reader=reader)
                records.append({
                    'idx': row.idx,
                    'frame': row.frame,
                    'method': f'easy_{pname}',
                    'text': text_easy,
                    'len': len(text_easy),
                    'non_empty': len(text_easy) > 0,
                    'looks_plate': looks_like_plate(text_easy)
                })
    return pd.DataFrame(records)

# Evaluar OCR en los recortes
df_ocr = evaluate_ocr(df_crops, PREPROCS, methods=None, max_samples=50)
df_ocr.head()


In [ ]:

    # Resumen por método
    summary = df_ocr.groupby('method').agg(
        total=('text', 'count'),
        non_empty_pct=('non_empty', lambda x: 100 * x.sum() / len(x)),
        looks_plate_pct=('looks_plate', lambda x: 100 * x.sum() / len(x))
    ).sort_values(by='non_empty_pct', ascending=False)
    display(summary)

    # Mostrar algunos ejemplos con su OCR
    def show_ocr_examples(df_crops: pd.DataFrame, df_ocr: pd.DataFrame, method: str, n: int = 4, figsize=(12, 6)):
        """
        Muestra n ejemplos para un método de OCR dado.
        """
        subset = df_ocr[df_ocr['method'] == method].copy()
        subset = subset.sort_values(by='looks_plate', ascending=False)
        subset = subset.head(n)
        fig, axes = plt.subplots(1, n, figsize=figsize)
        for ax, (_, row) in zip(axes, subset.iterrows()):
            crop = df_crops.loc[df_crops['idx'] == row['idx'], 'crop'].values[0]
            ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            ax.axis('off')
            ax.set_title(f"{method}
Pred: {row['text']}")
        plt.tight_layout()
        plt.show()

    # Mostrar ejemplos de EasyOCR con CLAHE
    if not df_ocr.empty:
        show_ocr_examples(df_crops, df_ocr, method='easy_clahe', n=4)
